# Verifying Factual Accuracy in AI-Generated News Summaries

## Introduction

In this exercise, we will check how accurate AI-generated news summaries are. We'll use a compact language model to create summaries and then check them against a fact database.

In [ ]:
!pip install transformers torch pandas nltk

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import string
import warnings

# Download necessary NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## Step 1: Set up the summarization model

We'll use a compact model for text summarization:

In [2]:
# Ignore TypedStorage is deprecated
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings("ignore", message="TypedStorage is deprecated")

model_name = "sshleifer/distilbart-cnn-6-6"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KarynaOhol1\.cache\huggingface\hub\models--sshleifer--distilbart-cnn-6-6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/460M [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/460M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

## Step 2: Create a function for summarization

In [3]:
def generate_summary(text):
    inputs = tokenizer([text], max_length=1024, return_tensors='pt', truncation=True)
    summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=150, early_stopping=True)
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

## Step 3: Create a simple fact database

In [4]:
fact_database = {
    "COVID-19": ["COVID-19 is caused by the SARS-CoV-2 virus", "The first cases were reported in Wuhan, China"],
    "Climate Change": ["Global temperatures are rising", "Human activities contribute to climate change"],
    "Artificial Intelligence": ["AI can perform tasks that typically require human intelligence", "Machine learning is a subset of AI"]
}

## Step 4: Create a function to check factual accuracy

In [5]:
def preprocess_text(text):
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    words = nltk.word_tokenize(text.lower())
    return ' '.join([lemmatizer.lemmatize(word) for word in words if word not in stop_words and word not in string.punctuation])

def check_fact_accuracy(summary, topic):
    summary_processed = preprocess_text(summary)
    facts = fact_database.get(topic, [])
    accuracy_scores = []
    
    for fact in facts:
        fact_processed = preprocess_text(fact)
        fact_words = set(fact_processed.split())
        summary_words = set(summary_processed.split())
        common_words = fact_words.intersection(summary_words)
        accuracy = len(common_words) / len(fact_words) if fact_words else 0
        accuracy_scores.append(accuracy)
    
    return sum(accuracy_scores) / len(accuracy_scores) if accuracy_scores else 0

## Step 5: Test the summarization and fact-checking

In [6]:
test_articles = {
    "COVID-19": "The COVID-19 pandemic, also known as the coronavirus pandemic, is an ongoing global pandemic of coronavirus disease 2019 (COVID-19) caused by severe acute respiratory syndrome coronavirus 2 (SARS-CoV-2). The virus was first identified in December 2019 in Wuhan, China. The World Health Organization declared a Public Health Emergency of International Concern on 30 January 2020, and later declared a pandemic on 11 March 2020. As of 9 June 2023, more than 689 million cases and 6.88 million deaths have been confirmed, making it one of the deadliest pandemics in history.",
    "Climate Change": "Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, such as through variations in the solar cycle. But since the 1800s, human activities have been the main driver of climate change, primarily due to burning fossil fuels like coal, oil and gas. Burning fossil fuels generates greenhouse gas emissions that act like a blanket wrapped around the Earth, trapping the sun's heat and raising temperatures. Examples of greenhouse gas emissions that are causing climate change include carbon dioxide and methane. These come from using gasoline for driving a car or coal for heating a building, for example. Clearing land and forests can also release carbon dioxide. Landfills for garbage are a major source of methane emissions. Energy, industry, transport, buildings, agriculture and land use are among the main emitters.",
    "Artificial Intelligence": "Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to natural intelligence displayed by animals including humans. AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals. The term 'artificial intelligence' had previously been used to describe machines that mimic and display 'human' cognitive skills that are associated with the human mind, such as 'learning' and 'problem-solving'. This definition has since been rejected by major AI researchers who now describe AI in terms of rationality and acting rationally, which does not limit how intelligence can be articulated."
}

for topic, article in test_articles.items():
    summary = generate_summary(article)
    accuracy = check_fact_accuracy(summary, topic)
    print(f"Topic: {topic}")
    print(f"Original article: {article[:100]}...")
    print(f"Generated summary: {summary}")
    print(f"Factual accuracy score: {accuracy:.2f}\n")

Topic: COVID-19
Original article: The COVID-19 pandemic, also known as the coronavirus pandemic, is an ongoing global pandemic of coro...
Generated summary:  The COVID-19 pandemic is an ongoing global pandemic of coronavirus disease 2019 . The virus was first identified in December 2019 in Wuhan, China . More than 689 million cases and 6.88 million deaths have been confirmed, making it one of the deadliest pandemics in history .
Factual accuracy score: 0.65

Topic: Climate Change
Original article: Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be ...
Generated summary: Climate change refers to long-term shifts in temperatures and weather patterns . Since the 1800s, human activities have been the main driver of climate change . Burning fossil fuels generates greenhouse gas emissions that act like a blanket wrapped around the Earth, trapping the sun's heat and raising temperatures .
Factual accuracy score: 0.57

Topic: Artificial Intellig

## Step 6: Analyze the results

Look at the summaries and accuracy scores. Answer these questions:

1. How well did the AI model summarize each article?

    - **COVID-19** — Good summary. Captured the key facts: pandemic name, origin in Wuhan (December 2019),and the death/case statistics. Stayed close to the original text.
    - **Climate Change** — Decent summary. Covered the main cause (human activities, fossil fuels) and the greenhouse effect mechanism, but omitted several emission sources mentioned in the original.
    - **Artificial Intelligence** — Weakest summary. Focused on the definition debate rather than explaining what AI actually is. Missed the most important introductory fact.

2. Which topic had the highest factual accuracy score? Why do you think this is?

    - COVID-19 scored highest (0.65) — because its fact database keywords ("SARS-CoV-2 virus","Wuhan, China") directly appeared in the summary.

3. Are there any important facts from the original articles that are missing in the summaries?

    - **COVID-19** — Missing: WHO declaration dates (January 30 and March 11, 2020), the term "Public Health Emergency of International Concern".
    - **Climate Change** — Missing: specific emission sources (landfills, agriculture, transport, buildings), mention of carbon dioxide and methane by name.
    - **AI** — Missing: the core definition *"intelligence demonstrated by machines"*, and any mention of ML which is in the fact database.


4. Did you notice any inaccuracies or misleading information in the summaries?

    - **AI summary** is potentially misleading — it emphasizes that the original definition "has been rejected", which could give the impression AI is poorly understood or disputed,without providing a clear alternative definition to the reader.


## Step 7: Improve the fact-checking function

Our current fact-checking function is quite simple. How could we improve it? Here are some ideas:

1. Use more advanced natural language processing techniques to understand context and meaning, not just word matching.
2. Expand the fact database with more detailed and nuanced information.
3. Implement a system to verify facts from reliable online sources in real-time.
4. Use a pre-trained model for natural language inference to check if the summary entails the facts.

Try implementing one of these improvements or come up with your own idea.

In [8]:
# Option 4: Improved Fact-Checking with Natural Language Inference (NLI)

#Instead of simple word matching, use a pre-trained NLI model to check whether the summary supports each fact from the database. Also, it handles paraphrasing — the summary doesn't need to use the exact same words as the fact db.

from transformers import pipeline

# Load a pre-trained NLI model
nli_model = pipeline("text-classification", model="cross-encoder/nli-deberta-v3-small")


def check_fact_accuracy_nli(summary, topic):
    facts = fact_database.get(topic, [])
    entailment_scores = []

    for fact in facts:
        # NLI: does the summary (premise) entail the fact (hypothesis)?
        result = nli_model(f"{summary} [SEP] {fact}")
        label = result[0]['label']
        score = result[0]['score']

        # Score 1.0 if entailment, 0.0 if contradiction, 0.5 if neutral
        if label == 'entailment':
            entailment_scores.append(score)
        elif label == 'neutral':
            entailment_scores.append(0.5 * score)
        else:  # contradiction
            entailment_scores.append(0.0)

    return sum(entailment_scores) / len(entailment_scores) if entailment_scores else 0


config.json: 0.00B [00:00, ?B/s]

C:\Users\KarynaOhol1\PycharmProjects\GenAI_for_DQE\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KarynaOhol1\.cache\huggingface\hub\models--cross-encoder--nli-deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/568M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [9]:
# Compare old vs new scores
for topic, article in test_articles.items():
    summary = generate_summary(article)
    old_score = check_fact_accuracy(summary, topic)
    new_score = check_fact_accuracy_nli(summary, topic)
    print(f"{topic:<25} {old_score:>12.2f} {new_score:>12.2f}")

COVID-19                          0.65         0.69
Climate Change                    0.57         0.74
Artificial Intelligence           0.39         0.45


## Step 8: Reflection and discussion

Think about these questions:

1. What are the challenges in automatically verifying the factual accuracy of AI-generated content?

    - Ambiguity in language and paraphrasing can make it hard to match facts. The summary may express the same fact in different words;Simple word matching (COVID-19: 0.65)\NLI (0.69) handles it better.
    - Incomplete coverage from fact database is exhaustive; AI may generate correct facts that simply aren't in the database

2. How might the summarization model introduce biases or inaccuracies?
    - The model may prioritize certain information over others, leading to an unbalanced summary. The training data for the model may contain biases that are reflected in the summaries it generates. The model may over-represent certain perspectives if its training  was not balanced across sources.
3. What are the potential consequences of using AI-generated news summaries without proper fact-checking?
    - Misinformation can spread quickly if summaries contain inaccuracies. Readers may form opinions based on incorrect information, which can have real-world consequences. It can damage the credibility of news sources and erode public trust in media.
4. How can we balance the need for concise summaries with the need for factual accuracy?
    - We can generate a summary, then run NLI fact-checking and append any missing critical facts as a footnote rather than expanding the full summary.
    - Set a minimum accuracy threshold before publishing.

## Bonus tasks

1. Try the summarization and fact-checking with articles on different topics or from different sources.
2. Implement a system to highlight which specific facts from the database are present or missing in each summary.
3. Explore how changing the summarization model's parameters (like max_length or num_beams) affects the factual accuracy of the summaries.
4. Discuss the ethical implications of using AI for news summarization and the importance of transparency in AI-generated content.

# please put your answer here